# Tutorial 7: PPI Context Embeddings

This tutorial no longer teaches direct loading of precomputed PPI matrices.
To keep tutorials centered on one public embedding API, PPI information is
first represented as biological text context and then embedded with
`BioEmbedder.embed(...)`.

If a dedicated PPI matrix backend is added to `BioEmbedder.embed(...)`, it
can replace the text-context model below without changing the tutorial's
public API shape.


In [ ]:
from embpy import BioEmbedder
from embpy.resources import GeneAnnotator

RUN_EMBEDDING = False
embedder = BioEmbedder(device="auto", organism="human")
annotator = GeneAnnotator(organism="human")


## 1. Build PPI-aware text contexts


In [ ]:
genes = ["TP53", "BRCA1", "EGFR", "KRAS", "MYC"]

contexts = []
for gene in genes:
    partners = annotator.get_protein_interactions(gene)[:10]
    partner_names = [p.get("preferredName_B") or p.get("stringId_B") for p in partners]
    partner_text = ", ".join(x for x in partner_names if x) or "no high-confidence partners found"
    contexts.append(f"{gene} protein interaction context: {partner_text}.")

print(contexts[0])


## 2. Embed those contexts through the unified API


In [ ]:
if RUN_EMBEDDING:
    ppi_context = embedder.embed(
        contexts,
        entity_type="text",
        model="minilm_l6_v2",
        output="anndata",
        key="X_ppi_context_minilm",
    )
    ppi_context.obs["gene"] = genes
    print(ppi_context.obsm["X_ppi_context_minilm"].shape)


## 3. Export context embeddings


In [ ]:
if RUN_EMBEDDING:
    embedder.embed(
        contexts,
        entity_type="text",
        model="minilm_l6_v2",
        output="table",
        path="ppi_context_embeddings.csv",
        fmt="csv",
    )
